# GPU Hours Estimation for Breast Cancer Detection NSGA-III Optimization

This notebook provides **real estimates** of GPU hours required for the multi-objective optimization study.

## Overview

The optimization uses **NSGA-III** to optimize 5 continuous hyperparameters across 4 objectives:
- **Each evaluation** trains a full ResNet152 model from scratch
- **Population size**: 20 (default)
- **Generations**: 50 (default)
- **Total evaluations**: 20 × 50 = **1,000 full training runs**

## Dataset Information
- Total images: 1,000 (500 breasts, 2 views each)
- Training split: ~800 images
- Validation split: ~200 images
- Class distribution: 25% malignant, 75% benign

## Training Configuration
- Model: ResNet152 (ImageNet pretrained)
- Batch size: 4
- Max epochs: 100
- Early stopping patience: 10 epochs (monitors PR-AUC)
- Optimizer: AdamW

In [ ]:
import sys
import os
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# Add breast_cancer_detection to path
sys.path.insert(0, 'breast_cancer_detection')

from breast_cancer_detection.src.models import build_resnet152
from breast_cancer_detection.src.preprocessing import MammographyPreprocessor
from breast_cancer_detection.src.datasets import VinDRMammoBinaryDataset, create_breast_level_splits
from breast_cancer_detection.src.augmentations import get_augmentation
from torch.utils.data import DataLoader

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

## Step 1: Benchmark Single Epoch Training Time

We'll train for a few epochs to get a realistic estimate of time per epoch.

In [ ]:
# Configuration
VINDR_IMAGES_ROOT = Path(r"C:\Users\HP\Downloads\New Project\data\vindr-mammo\images")
VINDR_CSV = Path(r"C:\Users\HP\Downloads\New Project\stratified_selection.csv")
BATCH_SIZE = 4
NUM_WORKERS = 2
RANDOM_SEED = 42
BENCHMARK_EPOCHS = 3  # Number of epochs to benchmark

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load dataset
print("Loading dataset...")
preprocessor = MammographyPreprocessor()

full_dataset = VinDRMammoBinaryDataset(
    images_root=str(VINDR_IMAGES_ROOT),
    csv_file=str(VINDR_CSV),
    preprocessor=preprocessor,
    transform=None
)

print(f"Total samples in dataset: {len(full_dataset)}")

# Create breast-level split
train_dataset, val_dataset = create_breast_level_splits(
    dataset=full_dataset,
    train_ratio=0.8,
    random_state=RANDOM_SEED,
    stratify=True
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

In [ ]:
# Create model and dataloaders for benchmarking
print("\nCreating model and dataloaders...")

# Use typical hyperparameters for benchmark
model = build_resnet152(
    pretrained=True,
    dropout=0.2,
    unfreeze_fraction=0.5
).to(device)

# Add augmentation to training dataset
augmentation = get_augmentation(strength=0.5)

from torch.utils.data import Subset
base_dataset = train_dataset.dataset
train_indices = train_dataset.indices

train_dataset_aug = VinDRMammoBinaryDataset(
    images_root=base_dataset.images_root,
    csv_file=None,
    preprocessor=base_dataset.preprocessor,
    transform=augmentation,
    samples=base_dataset.samples
)
train_dataset_aug = Subset(train_dataset_aug, train_indices)

train_loader = DataLoader(
    train_dataset_aug,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"Training batches per epoch: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

In [ ]:
# Benchmark training time per epoch
print(f"\n{'='*60}")
print(f"BENCHMARKING {BENCHMARK_EPOCHS} EPOCHS")
print(f"{'='*60}\n")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = torch.nn.BCEWithLogitsLoss()

epoch_times = []
val_times = []

for epoch in range(BENCHMARK_EPOCHS):
    # Training phase
    model.train()
    epoch_start = time.time()
    
    for batch_idx, (imgs, labels) in enumerate(train_loader):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.float().to(device, non_blocking=True)
        
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        if batch_idx % 50 == 0:
            print(f"  Epoch {epoch+1}/{BENCHMARK_EPOCHS} - Batch {batch_idx}/{len(train_loader)} - Loss: {loss.item():.4f}")
    
    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)
    
    # Validation phase
    model.eval()
    val_start = time.time()
    
    with torch.no_grad():
        val_loss = 0.0
        for imgs, labels in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.float().to(device, non_blocking=True)
            logits = model(imgs)
            loss = criterion(logits, labels)
            val_loss += loss.item()
    
    val_time = time.time() - val_start
    val_times.append(val_time)
    
    print(f"\nEpoch {epoch+1} completed:")
    print(f"  Training time: {epoch_time:.2f}s")
    print(f"  Validation time: {val_time:.2f}s")
    print(f"  Total time: {epoch_time + val_time:.2f}s\n")

avg_epoch_time = np.mean(epoch_times)
avg_val_time = np.mean(val_times)
avg_total_time = avg_epoch_time + avg_val_time

print(f"\n{'='*60}")
print(f"BENCHMARK RESULTS")
print(f"{'='*60}")
print(f"Average training time per epoch: {avg_epoch_time:.2f}s ({avg_epoch_time/60:.2f} min)")
print(f"Average validation time per epoch: {avg_val_time:.2f}s ({avg_val_time/60:.2f} min)")
print(f"Average total time per epoch: {avg_total_time:.2f}s ({avg_total_time/60:.2f} min)")
print(f"{'='*60}\n")

## Step 2: Estimate Epochs Until Early Stopping

Based on empirical observations and early stopping with patience=10:
- Minimum epochs: ~15-20 (if converges quickly)
- Average epochs: ~30-40 (typical case)
- Maximum epochs: 100 (if no early stopping)

We'll use conservative estimates for different scenarios.

In [ ]:
# Estimate epochs based on different scenarios
scenarios = {
    "Optimistic (Fast Convergence)": {
        "avg_epochs": 20,
        "description": "Model converges quickly, early stopping kicks in early"
    },
    "Realistic (Typical)": {
        "avg_epochs": 35,
        "description": "Average case with some hyperparameters converging slowly"
    },
    "Conservative (Slow Convergence)": {
        "avg_epochs": 50,
        "description": "Many evaluations require longer training"
    },
    "Worst Case (No Early Stop)": {
        "avg_epochs": 100,
        "description": "All evaluations run to maximum epochs"
    }
}

# Robustness evaluation overhead
# Robustness is evaluated every 5 epochs, so approximately 20% overhead
ROBUSTNESS_OVERHEAD = 1.2  # 20% overhead for robustness evaluations

print("Epoch Scenarios:")
print(f"{'='*80}")
for name, scenario in scenarios.items():
    print(f"\n{name}:")
    print(f"  Average epochs: {scenario['avg_epochs']}")
    print(f"  Description: {scenario['description']}")
print(f"\n{'='*80}")

## Step 3: Calculate Total GPU Hours for NSGA-III

Total GPU time = (# evaluations) × (avg epochs per evaluation) × (time per epoch) × (overhead factor)

In [ ]:
def calculate_gpu_hours(pop_size, n_generations, avg_epochs, time_per_epoch, overhead=1.2):
    """
    Calculate total GPU hours for NSGA-III optimization.
    
    Args:
        pop_size: Population size
        n_generations: Number of generations
        avg_epochs: Average epochs per evaluation
        time_per_epoch: Time per epoch in seconds
        overhead: Overhead factor (for robustness eval, logging, etc.)
    
    Returns:
        Dictionary with timing estimates
    """
    total_evaluations = pop_size * n_generations
    time_per_evaluation = avg_epochs * time_per_epoch * overhead
    total_time_seconds = total_evaluations * time_per_evaluation
    
    return {
        "total_evaluations": total_evaluations,
        "time_per_evaluation_min": time_per_evaluation / 60,
        "time_per_evaluation_hours": time_per_evaluation / 3600,
        "total_time_seconds": total_time_seconds,
        "total_time_minutes": total_time_seconds / 60,
        "total_time_hours": total_time_seconds / 3600,
        "total_time_days": total_time_seconds / 86400
    }

# Default NSGA-III configuration
DEFAULT_POP_SIZE = 20
DEFAULT_N_GENERATIONS = 50

print(f"\n{'='*80}")
print(f"GPU HOURS ESTIMATION FOR NSGA-III OPTIMIZATION")
print(f"{'='*80}\n")
print(f"Configuration:")
print(f"  Population size: {DEFAULT_POP_SIZE}")
print(f"  Generations: {DEFAULT_N_GENERATIONS}")
print(f"  Time per epoch: {avg_total_time:.2f}s ({avg_total_time/60:.2f} min)")
print(f"  Overhead factor: {ROBUSTNESS_OVERHEAD}x\n")
print(f"{'='*80}\n")

results = {}

for scenario_name, scenario_params in scenarios.items():
    result = calculate_gpu_hours(
        pop_size=DEFAULT_POP_SIZE,
        n_generations=DEFAULT_N_GENERATIONS,
        avg_epochs=scenario_params["avg_epochs"],
        time_per_epoch=avg_total_time,
        overhead=ROBUSTNESS_OVERHEAD
    )
    results[scenario_name] = result
    
    print(f"\n{scenario_name}:")
    print(f"  {scenario_params['description']}")
    print(f"  Average epochs per evaluation: {scenario_params['avg_epochs']}")
    print(f"  Total evaluations: {result['total_evaluations']}")
    print(f"  Time per evaluation: {result['time_per_evaluation_hours']:.2f} hours")
    print(f"  " + "-" * 60)
    print(f"  TOTAL GPU TIME: {result['total_time_hours']:.1f} hours ({result['total_time_days']:.2f} days)")
    print(f"  " + "=" * 60)

print(f"\n{'='*80}\n")

## Step 4: Visualization of GPU Time Estimates

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Total GPU hours by scenario
scenario_names = list(results.keys())
gpu_hours = [results[name]['total_time_hours'] for name in scenario_names]

colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
bars1 = axes[0].barh(scenario_names, gpu_hours, color=colors)
axes[0].set_xlabel('Total GPU Hours', fontsize=12, fontweight='bold')
axes[0].set_title('GPU Hours Required by Scenario\n(Pop=20, Gen=50)', fontsize=13, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Add value labels on bars
for i, (bar, hours) in enumerate(zip(bars1, gpu_hours)):
    days = results[scenario_names[i]]['total_time_days']
    axes[0].text(hours + 5, bar.get_y() + bar.get_height()/2, 
                f'{hours:.1f}h\n({days:.2f} days)', 
                va='center', fontsize=10, fontweight='bold')

# Plot 2: Time breakdown per evaluation
time_per_eval = [results[name]['time_per_evaluation_hours'] for name in scenario_names]

bars2 = axes[1].barh(scenario_names, time_per_eval, color=colors)
axes[1].set_xlabel('Hours per Evaluation', fontsize=12, fontweight='bold')
axes[1].set_title('Time per Single Evaluation\n(Training + Validation + Robustness)', fontsize=13, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

# Add value labels
for bar, hours in zip(bars2, time_per_eval):
    axes[1].text(hours + 0.02, bar.get_y() + bar.get_height()/2, 
                f'{hours:.2f}h', 
                va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('gpu_hours_estimation.png', dpi=150, bbox_inches='tight')
print("\nVisualization saved as 'gpu_hours_estimation.png'")
plt.show()

## Step 5: Sensitivity Analysis - Different Population Sizes and Generations

In [ ]:
# Sensitivity analysis for different configurations
pop_sizes = [10, 20, 30, 40]
generations = [25, 50, 75, 100]

# Use realistic scenario (35 epochs average)
realistic_avg_epochs = scenarios["Realistic (Typical)"]["avg_epochs"]

print(f"\n{'='*80}")
print(f"SENSITIVITY ANALYSIS: GPU HOURS FOR DIFFERENT CONFIGURATIONS")
print(f"{'='*80}\n")
print(f"Assuming: {realistic_avg_epochs} avg epochs per evaluation (Realistic scenario)\n")

# Create dataframe for results
sensitivity_results = []

for pop in pop_sizes:
    for gen in generations:
        result = calculate_gpu_hours(
            pop_size=pop,
            n_generations=gen,
            avg_epochs=realistic_avg_epochs,
            time_per_epoch=avg_total_time,
            overhead=ROBUSTNESS_OVERHEAD
        )
        sensitivity_results.append({
            "Population Size": pop,
            "Generations": gen,
            "Total Evaluations": result['total_evaluations'],
            "GPU Hours": result['total_time_hours'],
            "GPU Days": result['total_time_days']
        })

df_sensitivity = pd.DataFrame(sensitivity_results)

# Pivot table for heatmap
pivot_hours = df_sensitivity.pivot(index="Population Size", columns="Generations", values="GPU Hours")
pivot_days = df_sensitivity.pivot(index="Population Size", columns="Generations", values="GPU Days")

print("GPU Hours Required:")
print(pivot_hours.to_string())
print(f"\nGPU Days Required:")
print(pivot_days.to_string())

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))

import matplotlib.cm as cm
from matplotlib.colors import Normalize

# Create heatmap
im = ax.imshow(pivot_hours.values, cmap='YlOrRd', aspect='auto')

# Set ticks
ax.set_xticks(np.arange(len(generations)))
ax.set_yticks(np.arange(len(pop_sizes)))
ax.set_xticklabels(generations)
ax.set_yticklabels(pop_sizes)

# Labels
ax.set_xlabel('Number of Generations', fontsize=12, fontweight='bold')
ax.set_ylabel('Population Size', fontsize=12, fontweight='bold')
ax.set_title('GPU Hours Required for NSGA-III Optimization\n(Realistic Scenario: 35 avg epochs)', 
             fontsize=14, fontweight='bold', pad=20)

# Add text annotations
for i in range(len(pop_sizes)):
    for j in range(len(generations)):
        hours = pivot_hours.values[i, j]
        days = pivot_days.values[i, j]
        text = ax.text(j, i, f'{hours:.0f}h\n({days:.1f}d)',
                      ha="center", va="center", color="black", fontsize=9, fontweight='bold')

# Colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('GPU Hours', rotation=270, labelpad=20, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('sensitivity_analysis_heatmap.png', dpi=150, bbox_inches='tight')
print("\nHeatmap saved as 'sensitivity_analysis_heatmap.png'")
plt.show()

## Step 6: Summary and Recommendations

In [ ]:
print(f"\n{'='*80}")
print(f"SUMMARY AND RECOMMENDATIONS")
print(f"{'='*80}\n")

realistic_result = results["Realistic (Typical)"]
optimistic_result = results["Optimistic (Fast Convergence)"]
conservative_result = results["Conservative (Slow Convergence)"]

print(f"Based on {BENCHMARK_EPOCHS} epochs of actual GPU benchmarking:\n")
print(f"Time per epoch: {avg_total_time:.2f}s ({avg_total_time/60:.2f} min)")
print(f"\nFor NSGA-III with Population=20, Generations=50:\n")
print(f"  Best case (fast convergence):")
print(f"    - {optimistic_result['total_time_hours']:.1f} GPU hours ({optimistic_result['total_time_days']:.2f} days)")
print(f"\n  Expected case (realistic):")
print(f"    - {realistic_result['total_time_hours']:.1f} GPU hours ({realistic_result['total_time_days']:.2f} days)")
print(f"\n  Worst case (slow convergence):")
print(f"    - {conservative_result['total_time_hours']:.1f} GPU hours ({conservative_result['total_time_days']:.2f} days)")

print(f"\n{'='*80}")
print(f"RECOMMENDATIONS:")
print(f"{'='*80}\n")

print(f"1. For initial experiments:")
print(f"   - Start with smaller population (10) and fewer generations (25)")
result_small = calculate_gpu_hours(10, 25, realistic_avg_epochs, avg_total_time, ROBUSTNESS_OVERHEAD)
print(f"   - Estimated time: {result_small['total_time_hours']:.1f} hours ({result_small['total_time_days']:.2f} days)")
print(f"   - This allows quick validation of the optimization setup")

print(f"\n2. For production runs:")
print(f"   - Use Population=20, Generations=50 (default)")
print(f"   - Budget: {realistic_result['total_time_hours']:.1f} hours ({realistic_result['total_time_days']:.2f} days)")
print(f"   - Consider using cloud GPU (e.g., A100, V100) for faster training")

print(f"\n3. To reduce GPU time:")
print(f"   - Use surrogate-assisted optimization (as implemented in run_nsga3_surrogate.py)")
print(f"   - This can reduce true evaluations by 50-70%")
result_surrogate = calculate_gpu_hours(20, 50, realistic_avg_epochs * 0.3, avg_total_time, ROBUSTNESS_OVERHEAD)
print(f"   - Estimated time with surrogate: {result_surrogate['total_time_hours']:.1f} hours ({result_surrogate['total_time_days']:.2f} days)")

print(f"\n4. Parallelization:")
print(f"   - NSGA-III evaluates population in batches")
print(f"   - With 4 GPUs in parallel: {realistic_result['total_time_days']/4:.2f} days")
print(f"   - With 8 GPUs in parallel: {realistic_result['total_time_days']/8:.2f} days")

print(f"\n{'='*80}\n")

## Step 7: Export Results to CSV

In [ ]:
# Export sensitivity analysis to CSV
df_sensitivity.to_csv('gpu_hours_sensitivity_analysis.csv', index=False)
print("Sensitivity analysis exported to 'gpu_hours_sensitivity_analysis.csv'")

# Export scenario comparison
scenario_comparison = []
for name, result in results.items():
    scenario_comparison.append({
        "Scenario": name,
        "Avg Epochs": scenarios[name]["avg_epochs"],
        "Total Evaluations": result["total_evaluations"],
        "Hours per Evaluation": round(result["time_per_evaluation_hours"], 2),
        "Total GPU Hours": round(result["total_time_hours"], 1),
        "Total GPU Days": round(result["total_time_days"], 2)
    })

df_scenarios = pd.DataFrame(scenario_comparison)
df_scenarios.to_csv('gpu_hours_scenario_comparison.csv', index=False)
print("Scenario comparison exported to 'gpu_hours_scenario_comparison.csv'")

print(f"\n{'='*80}")
print(f"ALL RESULTS EXPORTED SUCCESSFULLY")
print(f"{'='*80}\n")
print("Files generated:")
print("  - gpu_hours_estimation.png")
print("  - sensitivity_analysis_heatmap.png")
print("  - gpu_hours_sensitivity_analysis.csv")
print("  - gpu_hours_scenario_comparison.csv")